# Ridge Regression: Reproducible Development Set

Change the hyperparameters in the configuration cell, rerun the notebook, and compare the reported development MAE. The split stays fixed across runs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'train.csv').exists():
    DATA_DIR = DATA_DIR / 'whatmakestitle'

# Hyperparameters: change these values and rerun from this cell onward.
DEV_SIZE = 0.20
RANDOM_STATE = 42
RIDGE_ALPHA = 10.0
WORD_NGRAM_RANGE = (1, 2)
CHAR_NGRAM_RANGE = (3, 5)
MIN_DF = 1

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
X = train_df[['title', 'domain']]
y = train_df['interest']
train_idx, dev_idx = train_test_split(
    np.arange(len(train_df)), test_size=DEV_SIZE, random_state=RANDOM_STATE,
    stratify=y)
X_train, X_dev = X.iloc[train_idx], X.iloc[dev_idx]
y_train, y_dev = y.iloc[train_idx], y.iloc[dev_idx]
print(f'Train rows: {len(train_idx)} | dev rows: {len(dev_idx)}')

## Feature pipeline and development evaluation

In [ ]:
features = ColumnTransformer([
    ('title_words', TfidfVectorizer(ngram_range=WORD_NGRAM_RANGE, min_df=MIN_DF, sublinear_tf=True), 'title'),
    ('title_chars', TfidfVectorizer(analyzer='char', ngram_range=CHAR_NGRAM_RANGE, min_df=MIN_DF, sublinear_tf=True), 'title'),
    ('domain', OneHotEncoder(handle_unknown='ignore'), ['domain']),
])
model = Pipeline([
    ('features', features),
    ('regressor', Ridge(alpha=RIDGE_ALPHA)),
])
model.fit(X_train, y_train)
dev_predictions = np.clip(model.predict(X_dev), 1, 6)
dev_mae = mean_absolute_error(y_dev, dev_predictions)
print(f'Development MAE: {dev_mae:.4f}')

## Final fit and submission

After selecting hyperparameters using development MAE, fit the selected pipeline on all labeled rows.

In [ ]:
model.fit(X, y)
test_predictions = np.clip(model.predict(test_df[['title', 'domain']]), 1, 6)
submission = pd.DataFrame({'title_id': test_df['title_id'], 'interest': test_predictions})
submission_path = DATA_DIR / 'submission_ridge.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} predictions to {submission_path}')
submission.head()